# Notebook 02: GIS Multi-Criteria Decision Making (AHP-TOPSIS)

**Objective:** Compute Analytic Hierarchy Process (AHP) criteria weights with Consistency Ratio ($CR < 0.10$), perform TOPSIS ranking, and generate high-resolution suitability heatmaps for candidate site screening across Dhaka City Corporation zones.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.spatial.ahp_mcdm import calculate_ahp_weights, topsis_ranking, run_ahp_spatial_pipeline
from src.visualization.map_plots import plot_ahp_suitability_map

config_path = os.path.join(module_path, 'configs', 'default_config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

pairwise_matrix = config['ahp_mcdm']['pairwise_matrix']
weights, lambda_max, CI, CR = calculate_ahp_weights(pairwise_matrix)

print(f'Principal Eigenvalue (lambda_max): {lambda_max:.4f}')
print(f'Consistency Index (CI): {CI:.4f}')
print(f'Consistency Ratio (CR): {CR:.4f} (Valid: {CR < 0.10})')

for crit, w in zip(config['ahp_mcdm']['criteria'].keys(), weights):
    print(f'  - {crit:24s}: {w*100:.2f}%')

## 1. Candidate Site Screening & TOPSIS Closeness Ranking

In [ ]:
candidate_gdf, _, _ = run_ahp_spatial_pipeline(config_path)
display(candidate_gdf[['candidate_id', 'site_name', 'zone_name', 'ahp_suitability_score', 'topsis_score', 'topsis_rank']].head(10))

## 2. Visualizing Dhaka Spatial Suitability Heatmap

In [ ]:
landuse_path = os.path.join(module_path, 'data', 'raw', 'rajuk_dap_landuse.geojson')
landuse_gdf = gpd.read_file(landuse_path)
output_fig = os.path.join(module_path, 'results', 'figures', 'ahp_suitability_map.png')
plot_ahp_suitability_map(candidate_gdf, landuse_gdf, output_fig)

img = plt.imread(output_fig)
plt.figure(figsize=(10, 12))
plt.imshow(img)
plt.axis('off')
plt.show()